# Unveiling the Android App Market: Google Play Store Analysis
**Oasis Infobyte SIP · Data Analytics · Level 2 · Task 4**

**Goal:** clean real-world Play Store data, explore categories, ratings, size/installs and pricing, run sentiment analysis on user reviews, and finish with data-driven advice for a developer launching a new app.

**Datasets (Kaggle, "Google Play Store Apps"):** `googleplaystore.csv` and `googleplaystore_user_reviews.csv`. Place both next to this notebook. If either is missing, the notebook falls back to a **synthetic stand-in** so the code runs. Re-run with the real files for your final results and PNGs.

In [ ]:
import os, re, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RS = 42
os.makedirs("images", exist_ok=True)
def save(name):
    plt.tight_layout(); plt.savefig(f"images/{name}.png", dpi=150, bbox_inches="tight"); plt.show()

## 1. Load both datasets separately

In [ ]:
def make_synthetic_apps(n=6000, seed=RS):
    """Stand-in with the same schema as googleplaystore.csv. NOT real data."""
    r = np.random.default_rng(seed)
    cats = ["FAMILY","GAME","TOOLS","BUSINESS","MEDICAL","PRODUCTIVITY","FINANCE","LIFESTYLE","SPORTS","COMMUNICATION","PHOTOGRAPHY","HEALTH_AND_FITNESS","EDUCATION"]
    w = np.array([18,12,11,7,6,6,5,5,5,4,4,4,3], float); w /= w.sum()
    cat = r.choice(cats, n, p=w)
    paid = r.random(n) < np.where(np.isin(cat, ["MEDICAL","PRODUCTIVITY","GAME"]), .12, .06)
    inst = r.choice([1e3,1e4,1e5,1e6,1e7,1e8], n, p=[.15,.25,.28,.2,.09,.03])
    size = np.clip(r.lognormal(2.8, .9, n), 1, 100)
    rating = np.clip(r.normal(4.2, .5, n) + 0.002*size, 1, 5).round(1)
    df = pd.DataFrame({
        "App": [f"App {i}" for i in range(n)], "Category": cat, "Rating": rating,
        "Reviews": (inst * r.uniform(.005, .03, n)).astype(int).astype(str),
        "Size": [f"{s:.1f}M" if r.random() > .1 else "Varies with device" for s in size],
        "Installs": [f"{int(i):,}+" for i in inst], "Type": np.where(paid, "Paid", "Free"),
        "Price": np.where(paid, [f"${p:.2f}" for p in r.choice([.99,1.99,2.99,4.99,9.99,19.99], n)], "0"),
        "Content Rating": "Everyone", "Genres": cat.astype(str), "Last Updated": "January 7, 2018",
        "Current Ver": "1.0", "Android Ver": "4.1 and up"})
    df.loc[r.random(n) < .12, "Rating"] = np.nan
    return pd.concat([df, df.sample(120, random_state=RS)], ignore_index=True)   # duplicates, like the real file

def make_synthetic_reviews(apps, per_app=6, seed=RS):
    r = np.random.default_rng(seed)
    pos = ["Love this app, works great", "Excellent and very useful", "Amazing design, highly recommend", "Best app I have used, so easy"]
    neg = ["Terrible, keeps crashing", "Worst update ever, very annoying", "Useless and full of ads, I hate it", "Bad experience, waste of time"]
    neu = ["It is okay", "Does what it says", "Average app", "Fine for now"]
    rows = []
    for a, rt in zip(apps["App"].unique()[:1500], apps.drop_duplicates("App")["Rating"].fillna(4.0)[:1500]):
        p_pos = np.clip((rt - 2.5) / 3, .1, .85)
        for _ in range(per_app):
            k = r.choice(["p", "n", "u"], p=[p_pos, (1 - p_pos) * .6, (1 - p_pos) * .4])
            rows.append((a, r.choice({"p": pos, "n": neg, "u": neu}[k])))
    return pd.DataFrame(rows, columns=["App", "Translated_Review"])

SYNTHETIC = not (os.path.exists("googleplaystore.csv") and os.path.exists("googleplaystore_user_reviews.csv"))
if SYNTHETIC:
    print("WARNING: dataset files not found - using SYNTHETIC data. Results are not real.")
    apps = make_synthetic_apps(); reviews = make_synthetic_reviews(apps)
else:
    apps = pd.read_csv("googleplaystore.csv"); reviews = pd.read_csv("googleplaystore_user_reviews.csv")
print("apps:", apps.shape, "| reviews:", reviews.shape)
apps.head()

## 2. Data cleaning
The raw file stores numbers as text: `Installs` is `"10,000+"`, `Price` is `"$4.99"`, `Size` is `"19M"`/`"201k"`/`"Varies with device"`. We fix types, handle nulls and drop duplicates.

In [ ]:
raw_rows = len(apps)
print("Nulls before:\n", apps.isnull().sum()[lambda s: s > 0])
a = apps.copy()
# the real file has one known corrupted row where columns are shifted; drop rows with a non-numeric Reviews value
a = a[a["Reviews"].astype(str).str.replace(r"\D", "", regex=True) != ""]
a["Reviews"] = a["Reviews"].astype(str).str.replace(r"\D", "", regex=True).astype(int)
a["Installs"] = a["Installs"].astype(str).str.replace(r"[+,]", "", regex=True)
a = a[a["Installs"].str.isdigit()]; a["Installs"] = a["Installs"].astype(np.int64)
a["Price"] = a["Price"].astype(str).str.replace("$", "", regex=False).astype(float)
def size_mb(s):
    s = str(s)
    if s.endswith("M"): return float(s[:-1].replace(",", ""))
    if s.endswith("k"): return float(s[:-1].replace(",", "")) / 1024
    return np.nan                                       # "Varies with device" -> NaN
a["Size_MB"] = a["Size"].apply(size_mb)
a["Rating"] = pd.to_numeric(a["Rating"], errors="coerce")
a = a.drop_duplicates(subset="App", keep="first")     # same app listed several times
a = a.dropna(subset=["Rating"])                         # ratings are central to the analysis
a["Category"] = a["Category"].astype(str).str.replace("_", " ").str.title()
print(f"\nRows: {raw_rows:,} -> {len(a):,} after cleaning")
print(a.dtypes[["Reviews", "Installs", "Price", "Size_MB", "Rating"]])
apps_clean = a

**Cleaning decisions:** duplicates were removed by app name; apps with no rating were dropped since rating is the main target metric; "Varies with device" sizes were kept as NaN rather than guessed, so size analysis uses only apps that report a size.

## 3. Category analysis: which categories are most saturated?

In [ ]:
cat_counts = apps_clean["Category"].value_counts()
fig, ax = plt.subplots(figsize=(9, 7))
sns.barplot(y=cat_counts.index[:15], x=cat_counts.values[:15], palette="viridis", ax=ax)
ax.set_title("Top 15 categories by number of apps (most saturated)"); ax.set_xlabel("Number of apps")
save("01_category_distribution")
print((cat_counts / cat_counts.sum() * 100).round(1).head(5))

## 4. Ratings analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(apps_clean["Rating"], bins=30, kde=True, color="#4C78A8", ax=axes[0])
axes[0].axvline(apps_clean["Rating"].mean(), color="red", ls="--", label=f"Mean {apps_clean['Rating'].mean():.2f}"); axes[0].legend()
axes[0].set_title("Distribution of app ratings")
avg = apps_clean.groupby("Category")["Rating"].mean().sort_values().tail(15)
avg.plot(kind="barh", ax=axes[1], color="#54A24B"); axes[1].set_xlim(3.5, 5); axes[1].set_title("Average rating by category (top 15)")
save("02_ratings_analysis")

## 5. Size vs. installs

In [ ]:
s = apps_clean.dropna(subset=["Size_MB"])
corr = s[["Size_MB"]].assign(log_installs=np.log10(s["Installs"].clip(lower=1))).corr().iloc[0, 1]
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.scatter(s["Size_MB"], s["Installs"], alpha=.25, s=12, color="#4C78A8")
ax.set_yscale("log"); ax.set_xlabel("App size (MB)"); ax.set_ylabel("Installs (log scale)")
ax.set_title(f"Size vs. installs (correlation with log installs: {corr:.2f})")
save("03_size_vs_installs")

## 6. Pricing analysis: paid vs. free, price distribution, revenue estimate

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
tc = apps_clean["Type"].value_counts()
axes[0].pie(tc.values, labels=tc.index, autopct="%1.1f%%", colors=["#4C78A8", "#F58518"], startangle=90); axes[0].set_title("Free vs. paid apps")
paid = apps_clean[(apps_clean["Type"] == "Paid") & (apps_clean["Price"] > 0)]
sns.histplot(paid["Price"].clip(upper=30), bins=30, color="#F58518", ax=axes[1]); axes[1].set_title("Price distribution of paid apps (capped at $30)"); axes[1].set_xlabel("Price ($)")
paid = paid.assign(Revenue=paid["Price"] * paid["Installs"])     # rough upper-bound estimate: price x installs
rev = paid.groupby("Category")["Revenue"].sum().sort_values().tail(10) / 1e6
rev.plot(kind="barh", ax=axes[2], color="#E45756"); axes[2].set_title("Estimated revenue by category, top 10 ($M)"); axes[2].set_xlabel("Price x installs, $ millions")
save("04_pricing_analysis")

*Revenue is a rough upper-bound estimate (price x installs). It ignores refunds, discounts and Google's cut, and the install counts are bucketed lower bounds.*

## 7. Sentiment analysis on user reviews
We classify each review as positive, negative or neutral using **VADER** (best on short informal text, needs no training) with **TextBlob** as an alternative. The compound score thresholds are the standard +/-0.05.

In [ ]:
rv = reviews.dropna(subset=["Translated_Review"]).drop_duplicates().copy()
engine = None
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    _v = SentimentIntensityAnalyzer(); score = lambda t: _v.polarity_scores(t)["compound"]; engine = "VADER"
except ImportError:
    try:
        from textblob import TextBlob
        score = lambda t: TextBlob(t).sentiment.polarity; engine = "TextBlob"
    except ImportError:
        POS = {"love","great","excellent","amazing","best","easy","useful","good","recommend"}
        NEG = {"terrible","worst","annoying","useless","hate","bad","crashing","waste","ads"}
        score = lambda t: (sum(w in POS for w in re.findall(r"\w+", t.lower())) - sum(w in NEG for w in re.findall(r"\w+", t.lower()))) / 3
        engine = "FALLBACK keyword lexicon (install vaderSentiment for real use)"
print("Sentiment engine:", engine)
rv["score"] = rv["Translated_Review"].astype(str).apply(score)
rv["Sentiment_Label"] = np.where(rv["score"] > .05, "Positive", np.where(rv["score"] < -.05, "Negative", "Neutral"))
print(rv["Sentiment_Label"].value_counts(normalize=True).mul(100).round(1))

## 8. Sentiment by category

In [ ]:
m = rv.merge(apps_clean[["App", "Category"]].drop_duplicates("App"), on="App")
ct = pd.crosstab(m["Category"], m["Sentiment_Label"], normalize="index") * 100
ct = ct.reindex(columns=["Positive", "Neutral", "Negative"], fill_value=0).sort_values("Positive")
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), gridspec_kw={"width_ratios": [1, 2]})
vc = rv["Sentiment_Label"].value_counts().reindex(["Positive", "Neutral", "Negative"]).fillna(0)
axes[0].bar(vc.index, vc.values, color=["#54A24B", "#BAB0AC", "#E45756"]); axes[0].set_title("Overall review sentiment")
ct.tail(15).plot(kind="barh", stacked=True, ax=axes[1], color=["#54A24B", "#BAB0AC", "#E45756"]); axes[1].set_title("Sentiment mix by category (%)"); axes[1].set_xlabel("% of reviews")
save("05_sentiment_analysis")

## 9. Interactive visualisation (Plotly, optional)

In [ ]:
try:
    import plotly.express as px
    g = apps_clean.groupby("Category").agg(Apps=("App", "count"), AvgRating=("Rating", "mean"), TotalInstalls=("Installs", "sum")).reset_index()
    fig = px.scatter(g, x="Apps", y="AvgRating", size="TotalInstalls", color="Category", hover_name="Category",
                     title="Competition vs. quality by category (bubble = total installs)")
    fig.write_html("images/interactive_category_bubble.html"); fig.show()
except ImportError:
    print("plotly not installed -> skipped (pip install plotly)")

## 10. Conclusion: 3 data-driven insights for a new app developer
*Write these from your real-data run.* Suggested angles, each backed by a chart above:
1. **Saturation vs. opportunity:** compare category size (chart 01) with average rating (chart 02). A less crowded category with decent installs is easier to stand out in.
2. **Pricing:** most apps are free (chart 04). If you charge, see which categories support paid apps and the typical price band.
3. **User sentiment:** categories with a high share of negative reviews (chart 05) show unmet needs you could target with a better product.

In [ ]:
print("Using synthetic data" if SYNTHETIC else "Using real Play Store data")
print(f"Apps analysed: {len(apps_clean):,} | Mean rating: {apps_clean['Rating'].mean():.2f} | Free share: {(apps_clean['Type']=='Free').mean()*100:.1f}%")